# Memory Management — Assignment 2

Thirty fully solved problems progressing from **medium → challenging**. Run top-to-bottom. Exact byte counts and some garbage-collector counts vary by Python version/platform; questions mark those outputs as variable.

Each problem contains the concept, question, hint, approach, code, expected output, explanation, and takeaway.

In [1]:
import copy
import gc
import sys
import tracemalloc
import weakref
from collections import deque
from contextlib import contextmanager
from functools import lru_cache
from itertools import islice

## Question 1 — Medium

**Concept tested:** References and identity

**Question:** Prove that assignment aliases a list while slicing creates a new outer list.

**Hint:** Compare both `is` and `==`.

### Solution approach

Create one list, assign an alias, create a slice, then mutate the original.

In [2]:
original = [1, 2]
alias = original
sliced = original[:]
original.append(3)
print(alias is original, sliced is original)
print(alias, sliced)

True False
[1, 2, 3] [1, 2]


### Expected output

```text
True False
[1, 2, 3] [1, 2]
```

### Step-by-step explanation

1. Assignment copied the reference. 2. Slicing created a new list. 3. Appending changed the shared object only.

### What was learned

Identity and equality are different; shallow outer copies break only the outer alias.

## Question 2 — Medium

**Concept tested:** `id()` and object lifetime

**Question:** Record an object's identity before and after rebinding an immutable integer.

**Hint:** Use `+=` on an integer.

### Solution approach

Store the first ID, rebind by arithmetic, then compare.

In [3]:
value = 1000
before_id = id(value)
value += 1
print(value, before_id != id(value))

1001 True


### Expected output

```text
1001 True
```

### Step-by-step explanation

Integers are immutable, so `+=` binds `value` to another integer object.

### What was learned

`id()` identifies an object during its lifetime; immutable updates rebind names.

## Question 3 — Medium

**Concept tested:** Reference counting

**Question:** Show that adding an alias increases CPython's reference count and deleting it lowers the count.

**Hint:** Compare differences, not absolute values.

### Solution approach

Measure a baseline, add alias, delete alias, and calculate deltas.

In [4]:
item = []
baseline = sys.getrefcount(item)
alias = item
with_alias = sys.getrefcount(item)
del alias
after_delete = sys.getrefcount(item)
print(with_alias - baseline, after_delete == baseline)

1 True


### Expected output

```text
1 True
```

### Step-by-step explanation

`getrefcount` temporarily references its argument, but that extra reference appears in every measurement and cancels in the difference.

### What was learned

Use reference-count deltas as a teaching aid; notebook internals can affect absolute counts.

## Question 4 — Medium

**Concept tested:** Mutability

**Question:** Write a function that appends to a list and show that the caller sees the change.

**Hint:** Function parameters are references to objects.

### Solution approach

Mutate the received list in place and print the caller's list.

In [5]:
def add_tag(tags):
    tags.append("new")

labels = ["python"]
add_tag(labels)
print(labels)

['python', 'new']


### Expected output

```text
['python', 'new']
```

### Step-by-step explanation

The parameter and caller name reference the same mutable list.

### What was learned

Mutating an argument affects aliases; document or avoid surprising mutation.

## Question 5 — Medium

**Concept tested:** Safe mutable defaults

**Question:** Fix a function that would otherwise reuse one default list across calls.

**Hint:** Use `None` as a sentinel.

### Solution approach

Create a fresh list inside when the argument is `None`.

In [6]:
def collect(value, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(value)
    return bucket

print(collect(1), collect(2))

[1] [2]


### Expected output

```text
[1] [2]
```

### Step-by-step explanation

Default expressions are evaluated once at function definition. `None` lets each call create its own list.

### What was learned

Avoid mutable default arguments unless shared state is deliberate.

## Question 6 — Medium

**Concept tested:** Nested aliasing

**Question:** Build a 3×2 grid whose rows are independent.

**Hint:** Use a comprehension rather than list repetition.

### Solution approach

Construct each row separately, mutate one cell, and display the grid.

In [7]:
grid = [[0] * 2 for _ in range(3)]
grid[0][0] = 9
print(grid)

[[9, 0], [0, 0], [0, 0]]


### Expected output

```text
[[9, 0], [0, 0], [0, 0]]
```

### Step-by-step explanation

The comprehension evaluates the inner list expression three times.

### What was learned

`[[...]] * n` aliases rows; comprehensions create independent rows.

## Question 7 — Medium

**Concept tested:** Shallow copy

**Question:** Show why a shallow-copied dictionary still shares a nested list.

**Hint:** Use `copy.copy`.

### Solution approach

Copy the outer dictionary, mutate the original inner list, and compare.

In [8]:
original = {"scores": [10, 20]}
shallow = copy.copy(original)
original["scores"].append(30)
print(shallow)
print(shallow["scores"] is original["scores"])

{'scores': [10, 20, 30]}
True


### Expected output

```text
{'scores': [10, 20, 30]}
True
```

### Step-by-step explanation

Only the dictionary shell was copied. Its value still points to the same list.

### What was learned

Shallow copies are sufficient only when shared nested objects are acceptable.

## Question 8 — Medium

**Concept tested:** Deep copy

**Question:** Make an independent nested configuration and prove a nested mutation does not cross over.

**Hint:** Use `copy.deepcopy`.

### Solution approach

Deep-copy, mutate the copy's inner list, then print both.

In [9]:
config = {"roles": ["reader"]}
independent = copy.deepcopy(config)
independent["roles"].append("writer")
print(config)
print(independent)

{'roles': ['reader']}
{'roles': ['reader', 'writer']}


### Expected output

```text
{'roles': ['reader']}
{'roles': ['reader', 'writer']}
```

### Step-by-step explanation

`deepcopy` recursively duplicated the supported nested list.

### What was learned

Deep copy separates nested mutable state, but has time/memory cost.

## Question 9 — Medium

**Concept tested:** Shallow size

**Question:** Demonstrate that `getsizeof` does not include nested list contents.

**Hint:** Print the outer and its children separately.

### Solution approach

Measure outer size and sum direct child sizes.

In [10]:
nested = [[1, 2], [3, 4]]
outer = sys.getsizeof(nested)
children = sum(sys.getsizeof(row) for row in nested)
print("outer positive:", outer > 0)
print("children positive:", children > 0)

outer positive: True
children positive: True


### Expected output

```text
outer positive: True
children positive: True
```

### Step-by-step explanation

The outer list stores references; inner lists are separate objects with separate sizes.

### What was learned

`sys.getsizeof` is shallow and platform-dependent.

## Question 10 — Medium

**Concept tested:** Generators

**Question:** Sum one million integers without first building a million-item list.

**Hint:** Pass a generator expression to `sum`.

### Solution approach

Create values lazily and let `sum` consume them.

In [11]:
total = sum(number for number in range(1_000_000))
print(total)

499999500000


### Expected output

```text
499999500000
```

### Step-by-step explanation

Only the generator and current values need to be live, rather than one giant list container.

### What was learned

Streaming can reduce peak memory while producing the same result.

## Question 11 — Medium

**Concept tested:** Generator exhaustion

**Question:** Show that a generator is one-pass and explain how to repeat the computation.

**Hint:** Call `list()` twice, then recreate it.

### Solution approach

Consume, consume again, then construct a fresh generator.

In [12]:
gen = (x * 2 for x in range(3))
print(list(gen))
print(list(gen))
print(list(x * 2 for x in range(3)))

[0, 2, 4]
[]
[0, 2, 4]


### Expected output

```text
[0, 2, 4]
[]
[0, 2, 4]
```

### Step-by-step explanation

A generator stores suspended computation state. Once exhausted, it does not rewind.

### What was learned

Recreate generators or use a reusable iterable when multiple passes are required.

## Question 12 — Medium

**Concept tested:** Bounded streaming

**Question:** Return the first five squares from an infinite generator safely.

**Hint:** Use `itertools.islice`.

### Solution approach

Build an infinite generator function and slice five outputs.

In [13]:
def squares_forever():
    number = 0
    while True:
        yield number * number
        number += 1

print(list(islice(squares_forever(), 5)))

[0, 1, 4, 9, 16]


### Expected output

```text
[0, 1, 4, 9, 16]
```

### Step-by-step explanation

`islice` stops requesting values after five, so the infinite producer remains safe.

### What was learned

Consumers should bound infinite or unknown streams.

## Question 13 — Medium

**Concept tested:** Circular references

**Question:** Create an unreachable two-list cycle and ask the collector to find it.

**Hint:** Disable automatic cyclic GC briefly for a deterministic demonstration.

### Solution approach

Save GC state, disable it, create/delete cycle, collect, restore state.

In [14]:
was_enabled = gc.isenabled()
gc.disable()
a, b = [], []
a.append(b)
b.append(a)
del a, b
found = gc.collect()
if was_enabled:
    gc.enable()
print("cycle objects found:", found >= 2)

cycle objects found: True


### Expected output

```text
cycle objects found: True
```

### Step-by-step explanation

The lists referenced each other, but no program name could reach them. Cyclic GC detected the unreachable graph.

### What was learned

Reference counting alone cannot reclaim cycles.

## Question 14 — Medium

**Concept tested:** Weak references

**Question:** Build a tiny cache that does not keep objects alive.

**Hint:** Use `weakref.WeakValueDictionary`.

### Solution approach

Insert an instance, remove its strong reference, collect, and check the cache.

In [15]:
class Picture:
    pass

cache = weakref.WeakValueDictionary()
picture = Picture()
cache["cover"] = picture
print("before:", "cover" in cache)
del picture
gc.collect()
print("after:", "cover" in cache)

before: True
after: False


### Expected output

```text
before: True
after: False
```

### Step-by-step explanation

The weak dictionary observed the instance but did not own a strong reference.

### What was learned

Weak caches allow entries to disappear when no real owner remains.

## Question 15 — Medium

**Concept tested:** Object cleanup

**Question:** Create a context manager that always records cleanup, even when work raises.

**Hint:** Put cleanup in `finally`.

### Solution approach

Yield a resource from a context manager and catch the demonstration error outside.

In [16]:
events = []
@contextmanager
def resource():
    events.append("open")
    try:
        yield
    finally:
        events.append("close")

try:
    with resource():
        events.append("use")
        raise ValueError("boom")
except ValueError:
    pass
print(events)

['open', 'use', 'close']


### Expected output

```text
['open', 'use', 'close']
```

### Step-by-step explanation

The context manager's `finally` runs as the `with` block exits, including exceptional exits.

### What was learned

Context managers provide deterministic resource cleanup.

## Question 16 — Medium

**Concept tested:** `__slots__`

**Question:** Create a slotted class and prove arbitrary attributes cannot be added.

**Hint:** Declare only `x` and `y` in `__slots__`.

### Solution approach

Instantiate, attempt another attribute, and catch `AttributeError`.

In [17]:
class Point:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

p = Point(1, 2)
try:
    p.color = "red"
except AttributeError as error:
    print(type(error).__name__)

AttributeError


### Expected output

```text
AttributeError
```

### Step-by-step explanation

Without a normal instance `__dict__`, only declared slots are available.

### What was learned

Slots can reduce per-instance overhead but restrict dynamic attributes and affect inheritance/weakrefs.

## Question 17 — Challenging

**Concept tested:** Recursive deep size

**Question:** Write a recursive size estimator that does not double-count a shared list.

**Hint:** Track visited `id()` values.

### Solution approach

Return zero for already-seen objects and recurse through dictionaries/containers.

In [18]:
def deep_size(obj, seen=None):
    seen = set() if seen is None else seen
    if id(obj) in seen:
        return 0
    seen.add(id(obj))
    total = sys.getsizeof(obj)
    if isinstance(obj, dict):
        total += sum(deep_size(k, seen) + deep_size(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset)):
        total += sum(deep_size(x, seen) for x in obj)
    return total

shared = [1, 2]
print(deep_size({"a": shared, "b": shared}) > sys.getsizeof(shared))

True


### Expected output

```text
True
```

### Step-by-step explanation

The `seen` set turns the object graph into a one-visit traversal and prevents cycles/double counts.

### What was learned

Identity tracking is essential when traversing arbitrary object graphs.

## Question 18 — Challenging

**Concept tested:** Tracemalloc snapshots

**Question:** Measure the allocation increase caused by retaining a list.

**Hint:** Compare snapshots by line number.

### Solution approach

Start tracing, snapshot, allocate, snapshot again, inspect the largest difference.

In [19]:
tracemalloc.start()
before = tracemalloc.take_snapshot()
retained = [str(i) for i in range(5_000)]
after = tracemalloc.take_snapshot()
top = after.compare_to(before, "lineno")[0]
print("positive growth:", top.size_diff > 0)
del retained
tracemalloc.stop()

positive growth:

 True


### Expected output

```text
positive growth: True
```

### Step-by-step explanation

The second snapshot contains allocations still reachable through `retained`.

### What was learned

Snapshot comparison locates growth rather than guessing from shallow sizes.

## Question 19 — Challenging

**Concept tested:** Bounded cache

**Question:** Use an LRU cache that never retains more than four results.

**Hint:** Set `maxsize=4` and inspect `cache_info()`.

### Solution approach

Call with ten keys, then check current size.

In [20]:
@lru_cache(maxsize=4)
def cube(number):
    return number ** 3
for number in range(10):
    cube(number)
print(cube.cache_info().currsize)
cube.cache_clear()

4


### Expected output

```text
4
```

### Step-by-step explanation

Least-recently-used entries are evicted when the fifth distinct key arrives.

### What was learned

Caches need an explicit size/expiry policy.

## Question 20 — Challenging

**Concept tested:** Closure retention

**Question:** Show that a closure can keep a large object reachable after the outer function returns.

**Hint:** Inspect `__closure__`.

### Solution approach

Capture a list in an inner function and inspect the stored cell.

In [21]:
def make_reader():
    large = [0] * 10_000
    def size():
        return len(large)
    return size

reader = make_reader()
print(reader())
print(len(reader.__closure__[0].cell_contents))

10000
10000


### Expected output

```text
10000
10000
```

### Step-by-step explanation

The closure cell owns a reference to `large`, so returning the function extends the list's lifetime.

### What was learned

Callbacks and closures can accidentally retain large graphs.

## Question 21 — Challenging

**Concept tested:** Deque retention

**Question:** Keep only the latest three events from a stream.

**Hint:** Use `deque(maxlen=3)`.

### Solution approach

Append five values and display retained history.

In [22]:
history = deque(maxlen=3)
for event in range(5):
    history.append(event)
print(list(history))

[2, 3, 4]


### Expected output

```text
[2, 3, 4]
```

### Step-by-step explanation

Appending beyond `maxlen` evicts the oldest item automatically.

### What was learned

Bounded containers prevent history buffers from growing forever.

## Question 22 — Challenging

**Concept tested:** Chunk processing

**Question:** Compute a total from 10,000 values in chunks of 1,000 without one result list.

**Hint:** Yield slices/ranges and aggregate each chunk.

### Solution approach

Build a chunk generator and sum chunk totals.

In [23]:
def chunks(start, stop, size):
    for left in range(start, stop, size):
        yield range(left, min(left + size, stop))

total = sum(sum(chunk) for chunk in chunks(0, 10_000, 1_000))
print(total)

49995000


### Expected output

```text
49995000
```

### Step-by-step explanation

Only one small `range` description and aggregation state are needed at a time.

### What was learned

Chunking bounds peak working memory and fits file/database pipelines.

## Question 23 — Challenging

**Concept tested:** Memoryview

**Question:** Change two bytes in a `bytearray` without copying the buffer.

**Hint:** Slice a `memoryview` and assign bytes of equal length.

### Solution approach

Wrap the buffer, update a view slice, and print the original.

In [24]:
buffer = bytearray(b"hello")
view = memoryview(buffer)
view[1:3] = b"AI"
print(buffer)
view.release()

bytearray(b'hAIlo')


### Expected output

```text
bytearray(b'hAIlo')
```

### Step-by-step explanation

The view references the same underlying bytes; assignment updates the original buffer.

### What was learned

`memoryview` enables zero-copy access to compatible binary buffers.

## Question 24 — Challenging

**Concept tested:** Copy protocol

**Question:** Make `copy.copy()` duplicate a custom object's list instead of sharing it.

**Hint:** Implement `__copy__`.

### Solution approach

Construct a new instance with a sliced list.

In [25]:
class Playlist:
    def __init__(self, songs):
        self.songs = songs
    def __copy__(self):
        return type(self)(self.songs.copy())

first = Playlist(["A"])
second = copy.copy(first)
second.songs.append("B")
print(first.songs, second.songs)

['A'] ['A', 'B']


### Expected output

```text
['A'] ['A', 'B']
```

### Step-by-step explanation

The custom copy method defines exactly which nested state becomes independent.

### What was learned

Classes can control copy semantics instead of accepting generic behavior.

## Question 25 — Challenging

**Concept tested:** Weak finalizers

**Question:** Register cleanup without defining `__del__`.

**Hint:** Use `weakref.finalize` and force a collection.

### Solution approach

Attach a callback, delete the instance, collect, and inspect events.

In [26]:
events = []
class Job:
    pass
job = Job()
weakref.finalize(job, events.append, "cleaned")
del job
gc.collect()
print(events)

['cleaned']


### Expected output

```text
['cleaned']
```

### Step-by-step explanation

The finalizer runs when the object becomes unreachable and avoids placing cleanup logic in the class destructor.

### What was learned

`weakref.finalize` is often safer than `__del__`, though context managers remain best for prompt cleanup.

## Question 26 — Challenging

**Concept tested:** GC state restoration

**Question:** Write a context manager that temporarily disables cyclic GC and always restores its previous state.

**Hint:** Remember `gc.isenabled()` and restore in `finally`.

### Solution approach

Disable on entry, yield, and conditionally re-enable on exit.

In [27]:
@contextmanager
def paused_gc():
    was_enabled = gc.isenabled()
    gc.disable()
    try:
        yield
    finally:
        if was_enabled:
            gc.enable()

with paused_gc():
    print("inside:", gc.isenabled())
print("outside:", gc.isenabled())

inside: False
outside: True


### Expected output

```text
inside: False
outside: True
```

### Step-by-step explanation

The context manager preserves global interpreter state even if the body fails.

### What was learned

Temporary runtime tuning must be exception-safe and restore prior settings.

## Question 27 — Challenging

**Concept tested:** Identity-safe graph walk

**Question:** Count unique list objects in a cyclic graph without infinite recursion.

**Hint:** Keep a `seen` identity set.

### Solution approach

Stop when an ID has already been visited.

In [28]:
def count_lists(obj, seen=None):
    seen = set() if seen is None else seen
    if id(obj) in seen:
        return 0
    seen.add(id(obj))
    if not isinstance(obj, list):
        return 0
    return 1 + sum(count_lists(item, seen) for item in obj)

a, b = [], []
a.append(b); b.append(a)
print(count_lists(a))
del a, b
gc.collect()

2


2

### Expected output

```text
2
```

### Step-by-step explanation

Each list contributes once; the back-edge encounters an existing identity and stops.

### What was learned

Cycle-safe traversals need identity-based visited tracking.

## Question 28 — Challenging

**Concept tested:** Streaming file-like data

**Question:** Process newline records lazily and ignore blanks without building an intermediate list.

**Hint:** Use nested generator expressions.

### Solution approach

Strip each line lazily, filter empty strings, and sum parsed integers.

In [29]:
lines = iter(["10\n", "\n", "20\n", "30\n"])
clean = (line.strip() for line in lines)
values = (int(line) for line in clean if line)
print(sum(values))

60


### Expected output

```text
60
```

### Step-by-step explanation

Each stage requests one item from the previous stage, keeping the pipeline lazy.

### What was learned

Generator pipelines compose memory-efficient transformations.

## Question 29 — Challenging

**Concept tested:** Allocation peak

**Question:** Use `tracemalloc.get_traced_memory()` to confirm a temporary list raises the peak.

**Hint:** Measure before, during, and after deleting.

### Solution approach

Trace, allocate, collect metrics, delete and collect, then compare.

In [30]:
tracemalloc.start()
before_current, _ = tracemalloc.get_traced_memory()
temporary = [0] * 50_000
during_current, peak = tracemalloc.get_traced_memory()
del temporary
gc.collect()
after_current, _ = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(during_current > before_current)
print(peak >= during_current)
print(after_current < during_current)

True
True
True


### Expected output

```text
True
True
True
```

### Step-by-step explanation

The live list increases current traced memory; the peak remembers the high-water point; deletion makes current tracing fall.

### What was learned

Current and peak measurements answer different questions.

## Question 30 — Challenging

**Concept tested:** Practical leak repair

**Question:** Replace an unbounded event list with a bounded recorder class.

**Hint:** Store events in `deque(maxlen=limit)`.

### Solution approach

Encapsulate the limit and return a snapshot copy.

In [31]:
class EventRecorder:
    def __init__(self, limit):
        self._events = deque(maxlen=limit)
    def record(self, event):
        self._events.append(event)
    def snapshot(self):
        return list(self._events)

recorder = EventRecorder(3)
for event in ["a", "b", "c", "d"]:
    recorder.record(event)
print(recorder.snapshot())

['b', 'c', 'd']


### Expected output

```text
['b', 'c', 'd']
```

### Step-by-step explanation

The container evicts automatically, and the snapshot prevents callers from mutating internal storage.

### What was learned

Bound growth at the data-structure boundary.

# Assignment Revision Cheat Sheet

- Assignment aliases; explicit copying creates another object.
- `is` checks identity, `==` checks value equality.
- `getsizeof` is shallow; graph traversal must avoid cycles/double counting.
- Shallow copies share nested references; deep copies recursively duplicate supported state.
- Reference counting handles most CPython cleanup; cyclic GC handles unreachable cycles.
- Context managers make resource cleanup deterministic; weak references do not keep objects alive.
- Generators, chunks, bounded deques/caches, slots, and zero-copy views can reduce memory.
- Use `tracemalloc` snapshots/current/peak measurements before optimizing.